<a href="https://colab.research.google.com/github/harish-30-git/bpo-customer-negotiator-agent/blob/main/Customer_negotiator_agent_for_BPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!pip install -q langgraph langchain langchain-ollama langchain-core

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from typing import TypedDict, Annotated
import operator

print("Setup complete!")

Setup complete!


In [19]:
llm = ChatOllama(
    model="qwen2.5:7b-instruct-q5_K_M",
    temperature=0.25,
    num_ctx=16384,
)

@tool
def get_customer_info(customer_id: str) -> str:
    """Fetch customer account details."""
    fake_db = {
        "CUST001": "Name: Harish, Bill: ₹4500 overdue by 45 days, Last payment: 45 days ago, Issue reported: 12 days ago, Plan: Postpaid Gold"
    }
    return fake_db.get(customer_id.upper(), "Customer not found.")

@tool
def calculate_offer(amount: float, discount_pct: float = 0, months_plan: int = 0) -> str:
    """Calculate final amount. Enforces max 25% discount."""
    if discount_pct > 25:
        return "Error: Maximum allowed discount is 25% as per policy."
    discounted = amount * (1 - discount_pct / 100)
    if months_plan > 0:
        emi = discounted / months_plan
        return f"Total after {discount_pct}% discount: ₹{discounted:.0f}. {months_plan}-month plan: ₹{emi:.0f}/month (no interest)."
    return f"Total after {discount_pct}% discount: ₹{discounted:.0f} (one-time)."

tools = [get_customer_info, calculate_offer]
llm_with_tools = llm.bind_tools(tools)

print("Tools and LLM ready!")

Tools and LLM ready!


In [20]:
system_prompt = """You are a professional, calm, and STRICT BPO Customer Negotiator for a telecom company.

Core Rules (NEVER break):
- Max discount: 25%. Never offer or calculate more.
- Full refund NOT allowed (bill is 45 days overdue).
- Always use tools before giving numbers or account info.
- Be empathetic but firm. Suggest win-win (discount + EMI).
- If customer repeatedly demands full refund or >25%, say: "I understand your frustration. I will escalate this to my supervisor."
- Keep replies short, natural, and professional.

Use get_customer_info when ID is given.
Use calculate_offer for any math/discount/plan."""

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    customer_id: str

# Nodes
def agent_node(state: AgentState):
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)

# Build Graph
workflow = StateGraph(AgentState)
workflow.add_node("agent", agent_node)
workflow.add_node("tools", tool_node)

workflow.set_entry_point("agent")
workflow.add_conditional_edges(
    "agent",
    lambda state: "tools" if state["messages"][-1].tool_calls else END,
    {"tools": "tools", END: END}
)
workflow.add_edge("tools", "agent")

app = workflow.compile()

print("LangGraph Agent built successfully!")

LangGraph Agent built successfully!


In [ ]:
print("=== Final BPO Customer Negotiator Agent ===\n")
print("Type your messages. Type 'exit' to quit.\n")

state = {"messages": [], "customer_id": "CUST001"}

while True:
    user_msg = input("You (Customer): ")
    if user_msg.lower() in ['exit', 'quit', 'bye']:
        print("Thank you for chatting. Goodbye!")
        break

    state["messages"].append(HumanMessage(content=user_msg))

    for output in app.stream(state):
        for node, value in output.items():
            if "messages" in value and value["messages"]:
                last_msg = value["messages"][-1]
                if isinstance(last_msg, AIMessage) and not last_msg.tool_calls:
                    print(f"\nNegotiator: {last_msg.content}\n")

    # Update state with new messages
    # (LangGraph handles it internally via the graph)
    print("=" * 70)